In [ ]:
import pandas as pd
import numpy as np
import warnings
import empyrical
import dai
import bigcharts 
import time 
warnings.filterwarnings('ignore')
print('导入包完成!')

In [ ]:
sql = f"""
    with t_tayler_reminder as (
        SELECT 
            date,
            trading_day,
            instrument,
            -- 简单收益率
            (close/LAG(close,1) over(partition by trading_day,instrument ORDER BY date) - 1)  as simple_ret,

            -- 连续复利收益率
            ln(close/LAG(close,1) over(partition by trading_day,instrument ORDER BY date)) as log_ret,

            -- 单复续利差 
            simple_ret - log_ret as  simple_log_spread,

            -- 泰勒残项 -- 二阶展开
            simple_log_spread * 2 - log_ret * log_ret as tayler_reminder
        FROM
            cn_stock_bar1m_derived_c    
    ),

    t_min_surge_intensity as (
        SELECT
            date::DATE::DATE as date,
            instrument,
            -- 计算泰勒残项 日平均值 = 日活跃度
            nanavg(tayler_reminder)  AS min_surge_intensity
        FROM
            t_tayler_reminder
        GROUP BY date::DATE,instrument
    ),

    t_daiy_surge_intensity as (
        SELECT 
            date,
            instrument,
            -- 振幅因子
            (high - low) / NULLIF(LAG(close) OVER (PARTITION BY instrument ORDER BY date), 0)  as raw_price_range,

            --日频简单收益率
            (close/LAG(close,1) over(partition by instrument ORDER BY date) - 1)  as daily_simple_ret,

            --日频复利收益率
            ln(close/LAG(close,1) over(partition by instrument ORDER BY date)) as daily_log_ret,

            -- 日频单复续利差 
            daily_simple_ret - daily_log_ret as  daily_simple_log_spread,

            -- 日频泰勒残项(活跃度) -- 二阶展开
            daily_simple_log_spread * 2 - daily_log_ret * daily_log_ret as daily_surge_intensity
        
        FROM
            cn_stock_bar1d
    ),

    t_moth2flame as (
        SELECT
            date,
            instrument,
            min_surge_intensity,
            min_surge_intensity_mean,
            raw_price_range,
            
            -- 通过分钟频数据 反转 振幅因子
            CASE 
                WHEN min_surge_intensity < min_surge_intensity_mean 
                THEN -raw_price_range 
                ELSE raw_price_range 
            END AS price_rang_1,

            -- 通过日频数据 反转 振幅因子
            daily_surge_intensity,
            daily_surge_intensity_mean,

            CASE 
                WHEN daily_surge_intensity < daily_surge_intensity_mean 
                THEN -raw_price_range 
                ELSE raw_price_range 
            END AS price_rang_2,
        FROM
        (
            SELECT 
                date,
                instrument,
                tbl.min_surge_intensity,
                ds.daily_surge_intensity,
                ds.raw_price_range,
                -- 横截面活跃度
                AVG(tbl.min_surge_intensity) over(partition by date) as  min_surge_intensity_mean,   

                -- 横截面日频活跃度
                AVG(ds.daily_surge_intensity) over(partition by date) as  daily_surge_intensity_mean, 
                
            FROM
                t_min_surge_intensity as tbl
            JOIN t_daiy_surge_intensity as ds using(date, instrument)
        )
    )
    
    SELECT
        date,
        instrument,
        NANAVG(price_rang_1) OVER (
            PARTITION BY instrument 
            ORDER BY date 
            ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
        ) AS _price_rang_1_mothmean,
        
        NANAVG(price_rang_2) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS _price_rang_2_mothmean,

        NANAVG(min_surge_intensity) OVER (
                PARTITION BY instrument 
                ORDER BY date 
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS _min_surge_intensity_mothmean,

        c_zscore(_price_rang_1_mothmean) + c_zscore(_price_rang_2_mothmean) as _price_rang,
        c_zscore(_price_rang) + c_zscore(_min_surge_intensity_mothmean) as factor
    FROM
        t_moth2flame
    ORDER BY
        date, instrument
    
"""


In [ ]:
def run(sql,shift_days):

    from bigquant import bigtrader, dai
    import pandas as pd
    from datetime import datetime, timedelta
    import numpy as np
    from sklearn.linear_model import LinearRegression

    def initialize(context: bigtrader.IContext):
        from bigtrader.finance.commission import PerOrder

        # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
        context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


        context.holding_days = 5
        context.target_hold_count = 50


    def befor_trading(context, data):
        pass
        
    def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):

        # 每 context.holding_days 个交易日调仓一次
        if context.trading_day_index % context.holding_days != 0:
            return




        # 获取当前日期
        ed = data.current_dt.strftime("%Y-%m-%d")
        
        date_obj = datetime.strptime(ed, "%Y-%m-%d")
    
        # 向前推10天
        n_days_ago = date_obj - timedelta(days=shift_days)
        tomorrow = date_obj + timedelta(days=1)
    
        # 转换回字符串格式
        sd = n_days_ago.strftime("%Y-%m-%d")
        ed2 = tomorrow.strftime("%Y-%m-%d")
        
        # 获取当日数据
        current_day_data = dai.query(sql,filters={'date':[sd,ed2]}).df()
        current_day_data['date']=pd.to_datetime(current_day_data['date']) 
        current_day_data = current_day_data[current_day_data.date==ed]

        # 取前10只
        current_day_data.sort_values(by='factor',inplace=True,ascending=True)

        current_day_data = current_day_data.head(context.target_hold_count)
        len_ = len(current_day_data)
        # 获取当日目标持有股票
        target_hold_instruments = set(current_day_data["instrument"])
        
        # 获取当前已持有股票
        current_hold_instruments = set(context.get_account_positions().keys())

        # 卖出不在目标持有列表中的股票
        for instrument in current_hold_instruments - target_hold_instruments:
            context.order_target_percent(instrument, 0)
            
        # 买入目标持有列表中的股票
        for instrument in target_hold_instruments - current_hold_instruments:
            context.order_target_percent(instrument, 1/len_)

    performance = bigtrader.run(
        market=bigtrader.Market.CN_STOCK,
        frequency=bigtrader.Frequency.DAILY,
        start_date='2021-12-31',  
        end_date='2025-12-31',  
        capital_base=3000000,     # 设置初始资金
        initialize=initialize,     # 传入初始化函数
        handle_data=handle_data,   # 传入数据处理函数
        before_trading_start = befor_trading,
        order_price_field_buy='open',
        order_price_field_sell='open',
    )

    # 渲染绩效报告，展示回测结果
    performance.render()

In [ ]:
run(sql,21)